In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document

C:\Users\DNFH3173\AppData\Local\Temp\ipykernel_27100\3112106335.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
docs=[
    Document(page_content="Langchain is a powerful framework for building language model applications.", metadata={"source": "test1.txt"}),
    Document(page_content="FAISS is a library for efficient similarity search and clustering of dense vectors.", metadata={"source": "test2.txt"}),
    Document(page_content="BM25 is a ranking function used by search engines to rank matching documents according to their relevance to a given search query.", metadata={"source": "test3.txt"}),
    Document(page_content="Ensemble retrievers combine multiple retrieval methods to improve search results.", metadata={"source": "test4.txt"}),
    Document(page_content="Eiffel Tower is a iconic landmark in Paris, France.", metadata={"source": "test5.txt"})
]

In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

c:\Users\DNFH3173\Desktop\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1698.10it/s]


In [7]:
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=3

#combine with ensemble retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
    )





In [8]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000220576D4590>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000220576D4980>, k=3)], weights=[0.5, 0.5])

In [9]:
query = "What is Langchain?"
results = hybrid_retriever.invoke(query)


for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Content: {doc.page_content}")
    print(f"Source: {doc.metadata['source']}")
    print()



Result 1:
Content: Langchain is a powerful framework for building language model applications.
Source: test1.txt

Result 2:
Content: FAISS is a library for efficient similarity search and clustering of dense vectors.
Source: test2.txt

Result 3:
Content: Eiffel Tower is a iconic landmark in Paris, France.
Source: test5.txt

Result 4:
Content: BM25 is a ranking function used by search engines to rank matching documents according to their relevance to a given search query.
Source: test3.txt

Result 5:
Content: Ensemble retrievers combine multiple retrieval methods to improve search results.
Source: test4.txt



### RAG Pipeline with hybrid retriever

In [14]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain


In [24]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("""
Answer the following question based on the context provided. If the answer is not contained within the context, say "I don't know."

Context:
{context}

Question:
{input}

""")

In [2]:

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("AZURE_FOUNDRY_MODEL"),
    base_url=os.getenv("AZURE_FOUNDRY_ENDPOINT"),
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
    temperature=0.7,
)

response = llm.invoke("What is the capital of France?")

print(response.content)


The capital of France is Paris.


In [26]:
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

rag_chain = create_retrieval_chain(retriever=hybrid_retriever, combine_docs_chain=document_chain)

In [27]:
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000220576D4590>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000220576D4980>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based on the context provided. If the answer is not contained within the context, say "I don\'t kn

In [29]:
query = {"input": "What is Langchain?"}
response = rag_chain.invoke(query)

print(response["answer"])

print("Sources:")
for i, doc in enumerate(response["context"]):
    print(f"Source {i+1}: {doc.page_content} (from {doc.metadata['source']})")

Langchain is a powerful framework for building language model applications.
Sources:
Source 1: Langchain is a powerful framework for building language model applications. (from test1.txt)
Source 2: FAISS is a library for efficient similarity search and clustering of dense vectors. (from test2.txt)
Source 3: Eiffel Tower is a iconic landmark in Paris, France. (from test5.txt)
Source 4: BM25 is a ranking function used by search engines to rank matching documents according to their relevance to a given search query. (from test3.txt)
Source 5: Ensemble retrievers combine multiple retrieval methods to improve search results. (from test4.txt)


In [18]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.schema import Document
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever




In [13]:
loader = PyPDFLoader("docs\\pdf\\document.pdf")
raw_doc=loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_doc)

docs

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-26T14:22:24-06:00', 'moddate': '2026-01-26T14:22:28-06:00', 'trapped': '/False', 'source': 'docs\\pdf\\document.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}, page_content='The Complete Guide \nto Building Skills \nfor Claude'),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-26T14:22:24-06:00', 'moddate': '2026-01-26T14:22:28-06:00', 'trapped': '/False', 'source': 'docs\\pdf\\document.pdf', 'total_pages': 33, 'page': 1, 'page_label': '2'}, page_content='Contents\nIntroduction 3\nFundamentals 4\nPlanning and design 7\nT esting and iteration 14\nDistribution and sharing 18\nPatterns and troubleshooting 21\nResources and references 28\n2'),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-26T14

In [14]:
query="what is ai agent?"

In [20]:
from langchain_classic.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever=vectorstore.as_retriever(search_kwargs={"k": 8})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2499.18it/s]


In [21]:
sparse_retriever = BM25Retriever.from_documents(docs)

sparse_retriever.k=3

#combine with ensemble retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
    )


In [38]:
query="what is skills of ai agent?"

retriever_results = hybrid_retriever.invoke(query)


In [39]:
formated_docs=[result.page_content for result in retriever_results[:10]]
formated_docs

['• Add skills to Messages API requests via the `container.skills` parameter \n• V ersion control and management through the Claude Console \n• Works with the Claude Agent SDK for building custom agents \nWhen to use skills via the API vs. Claude.ai: \nUse Case Best Surface\nEnd users interacting with skills directly Claude.ai / Claude Code\nManual testing and iteration during development Claude.ai / Claude Code\nIndividual, ad-hoc workflows Claude.ai / Claude Code',
 'content.\n---\nname: your-skill\ndescription: [--.]\n---\n# Your Skill Name\n-# Instructions\n--# Step 1: [First Major Step]\nClear explanation of what happens.\nExample:\n```bash\npython scripts/fetch_data.py --project-id PROJECT_ID\nExpected output: [describe what success looks like]\n(Add more steps as needed)\nExamples\nExample 1: [common scenario]\nUser says: "Set up a new marketing campaign"\nActions:\n1. Fetch existing campaigns via MCP \n2. Create new campaign with provided parameters',
 "Organization-level skill

In [40]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("Qwen/Qwen3-Reranker-0.6B")





Loading weights: 100%|██████████| 310/310 [00:01<00:00, 174.87it/s]


In [41]:
scores = model.predict([(query, page) for page in formated_docs])
print(scores)

[ 4.5     -5.625    4.75    -6.40625  1.5625  -6.5625  -0.9375   3.8125
  3.1875   2.8125 ]


In [42]:
formated_docs_with_scores = list(zip(formated_docs, scores))
formated_docs_with_scores.sort(key=lambda x: x[1], reverse=True)
for doc, score in formated_docs_with_scores:
    print(f"Score: {score}, Content: {doc}")    

Score: 4.75, Content: Organization-level skills:
• Admins can deploy skills workspace-wide (shipped December 18, 2025)
• Automatic updates
• Centralized management
An open standard
We've published Agent Skills as an open standard. Like MCP , we believe skills 
should be portable across tools and platforms - the same skill should work 
whether you're using Claude or other AI platforms. That said, some skills are 
designed to take full advantage of a specific platform's capabilities; authors can
Score: 4.5, Content: • Add skills to Messages API requests via the `container.skills` parameter 
• V ersion control and management through the Claude Console 
• Works with the Claude Agent SDK for building custom agents 
When to use skills via the API vs. Claude.ai: 
Use Case Best Surface
End users interacting with skills directly Claude.ai / Claude Code
Manual testing and iteration during development Claude.ai / Claude Code
Individual, ad-hoc workflows Claude.ai / Claude Code
Score: 3.8125, Cont